# 🏛️ Banco de Dados Relacional (SQL)
> **Missão:** Entender o impacto brutal de um Índice (B-Tree) ao ler dados gravados fisicamente no disco rígido.
> **Arquivo:** Os dados estão armazenados localmente no arquivo `banco_clientes.db`.

Abaixo, simularemos a diferença de buscar um usuário pela sua **Chave Primária** (que o banco sabe exatamente onde está) versus buscar por um texto solto (o que obriga o banco a ler o arquivo inteiro do começo ao fim).

In [1]:
import sqlite3
import os
import random
import warnings
warnings.filterwarnings('ignore')

db_path = 'banco_clientes.db'

# Verifica se o arquivo físico já existe no diretório
if not os.path.exists(db_path):
    print("⏳ Gerando 'banco_clientes.db' no disco pela primeira vez (isso pode levar alguns segundos)...")
    
    # Cria e conecta ao arquivo no disco
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE clientes (id INTEGER PRIMARY KEY, nome TEXT, idade INTEGER)')
    
    # Gerando 1 milhão de registros
    dados_sql = [(i, f"Usuario_{i}", random.randint(18, 80)) for i in range(1, 1000001)]
    cursor.executemany('INSERT INTO clientes VALUES (?, ?, ?)', dados_sql)
    
    conn.commit()
    conn.close()
    print("✅ Arquivo 'banco_clientes.db' criado e salvo com sucesso na pasta!")
else:
    print("✅ Arquivo 'banco_clientes.db' já existe. Pronto para as consultas!")

✅ Arquivo 'banco_clientes.db' já existe. Pronto para as consultas!


In [2]:
import time
import sqlite3
import pandas as pd
import ipywidgets as widgets
from ipywidgets import interact

# 1. Inicialização e Conexão
conn = sqlite3.connect("banco_clientes.db")

# 2. Função de Visualização (Isolada para cuidar apenas do visual)
def exibir_resultado(query: str, titulo: str, explicacao: str, tempo_ms: float, linhas_lidas: int, resultado: pd.DataFrame):
    """Formata e imprime os resultados da execução no estilo terminal."""
    
    print(f"\n{'=' * 80}")
    print("⚙️  PAINEL DE TESTES SQL")
    print(f"{'=' * 80}")

    print("\n💻 QUERY EXECUTADA")
    print(f"{'-' * 80}")
    print(query.strip()) 

    print("\n📊 RESUMO")
    print(f"{'-' * 80}")
    print(f"Tempo de execução : {tempo_ms:.2f} ms")
    print(f"Linhas lidas      : {linhas_lidas:,}".replace(",", "."))
    print(f"Resultado         : {titulo}")

    print("\n📝 O QUE ACONTECEU?")
    print(f"{'-' * 80}")
    print(explicacao)

    print("\n📋 RESULTADO DA CONSULTA")
    print(f"{'-' * 80}")
    
    if resultado.empty:
        print("Nenhum registro encontrado.")
    else:
        print(resultado.to_string(index=False))
        
    print(f"\n{'=' * 80}")


# 3. Lógica de Execução e Banco de Dados
def testar_sql(tipo_query: str):
    """Prepara a consulta, mede o tempo e busca os dados."""
    
    inicio = time.time()

    if tipo_query == "Eficiente (Busca por Chave Primária)":
        query = """
                SELECT * FROM clientes 
                WHERE id = 850000;
                """
        titulo = "🚀 Busca Indexada (B-Tree)"
        explicacao = (
            "O banco utilizou a chave primária como índice.\n"
            "Em vez de percorrer toda a tabela, ele navegou pela "
            "estrutura B-Tree e encontrou diretamente o registro."
        )
        linhas_lidas = 1
        
    else:
        query = """
                SELECT * FROM clientes 
                WHERE nome LIKE '%Usuario_850000%';
                """
        titulo = "🚨 Full Table Scan"
        explicacao = (
            "Não existe índice na coluna 'nome'.\n"
            "Por isso o banco precisou percorrer todas as 1.000.000 de linhas "
            "da tabela procurando o valor desejado."
        )
        linhas_lidas = 1_000_000

    # Executando a consulta
    resultado = pd.read_sql_query(query, conn)

    fim = time.time()
    tempo_ms = (fim - inicio) * 1000

    # Chamando a função de visualização
    exibir_resultado(query, titulo, explicacao, tempo_ms, linhas_lidas, resultado)


# 4. Interface Gráfica (Widgets)
opcoes_cenarios = [
    "Eficiente (Busca por Chave Primária)",
    "Ineficiente (Busca Textual sem Índice)"
]

interact(
    testar_sql,
    tipo_query=widgets.RadioButtons(
        options=opcoes_cenarios,
        description="Cenário:",
        layout={'width': 'max-content'}
    )
)

interactive(children=(RadioButtons(description='Cenário:', layout=Layout(width='max-content'), options=('Efici…

<function __main__.testar_sql(tipo_query: str)>